# Streaming

In [15]:
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI

def get_weather(city: str) -> str:
    """获取给定城市的天气。"""

    return f"It's always rainy in {city}!"

model = ChatOpenAI(model="Qwen/Qwen2.5-32B-Instruct")
agent = create_agent(
    model=model,
    tools=[get_weather],
)
for chunk in agent.stream(  # [!code highlight]
    {"messages": [{"role": "user", "content": "What is the weather in SF?"}]},
    stream_mode="updates",
):
    for step, data in chunk.items():
        print(f"step: {step}")
        print(f"content: {data['messages'][-1].content_blocks}")

step: model
content: [{'type': 'tool_call', 'name': 'get_weather', 'args': {'city': 'SF'}, 'id': '019b35524b8bcaf09d0908dbbc1610f5'}]
step: tools
content: [{'type': 'text', 'text': "It's always rainy in SF!"}]
step: model
content: [{'type': 'text', 'text': 'The weather in SF is always rainy!'}]


In [16]:
for token, metadata in agent.stream(  # [!code highlight]
    {"messages": [{"role": "user", "content": "What is the weather in SF?"}]},
    stream_mode="messages",
):
    print(f"node: {metadata['langgraph_node']}")
    print(f"content: {token.content_blocks}")
    print("\n")

node: model
content: []


node: model
content: [{'type': 'text', 'text': '<tool_call>'}]


node: model
content: [{'type': 'text', 'text': '\n'}]


node: model
content: [{'type': 'tool_call_chunk', 'id': '019b3552f583383475affcab5cc27bf7', 'name': 'get_weather', 'args': '', 'index': 0}]


node: model
content: [{'type': 'tool_call_chunk', 'id': None, 'name': None, 'args': ' {"', 'index': 0}]


node: model
content: [{'type': 'tool_call_chunk', 'id': None, 'name': None, 'args': 'city', 'index': 0}]


node: model
content: [{'type': 'tool_call_chunk', 'id': None, 'name': None, 'args': '":', 'index': 0}]


node: model
content: [{'type': 'tool_call_chunk', 'id': None, 'name': None, 'args': ' "', 'index': 0}]


node: model
content: [{'type': 'tool_call_chunk', 'id': None, 'name': None, 'args': 'SF', 'index': 0}]


node: model
content: [{'type': 'tool_call_chunk', 'id': None, 'name': None, 'args': '"}', 'index': 0}]


node: model
content: []


node: tools
content: [{'type': 'text', 'text': "It's

In [13]:
from langgraph.config import get_stream_writer  # [!code highlight]

def get_weather(city: str) -> str:
    """获取给定城市的天气。"""
    writer = get_stream_writer()  # [!code highlight]
    # 流式传输任何任意数据
    writer(f"Looking up data for city: {city}")
    writer(f"Acquired data for city: {city}")
    return f"It's always sunny in {city}!"

agent = create_agent(
    model=model,
    tools=[get_weather],
)

for chunk in agent.stream(
    {"messages": [{"role": "user", "content": "What is the weather in SF?"}]},
    stream_mode="custom"  # [!code highlight]
):
    print(chunk)

Looking up data for city: SF
Acquired data for city: SF


In [14]:
for stream_mode, chunk in agent.stream(  # [!code highlight]
    {"messages": [{"role": "user", "content": "What is the weather in SF?"}]},
    stream_mode=["updates", "custom"]
):
    print(f"stream_mode: {stream_mode}")
    print(f"content: {chunk}")
    print("\n")

stream_mode: updates
content: {'model': {'messages': [AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 19, 'prompt_tokens': 175, 'total_tokens': 194, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'Qwen/Qwen2.5-32B-Instruct', 'system_fingerprint': '', 'id': '019b352dddf1982b4745cbbc382629f7', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019b352d-d5b5-7690-8f8b-157d3e398878-0', tool_calls=[{'name': 'get_weather', 'args': {'city': 'SF'}, 'id': '019b352ddf29aa9c8dbd42e44876611e', 'type': 'tool_call'}], usage_metadata={'input_tokens': 175, 'output_tokens': 19, 'total_tokens': 194, 'input_token_details': {}, 'output_token_details': {}})]}}


stream_mode: custom
content: Looking up data for city: SF


stream_mode: custom
content: Acquired data for city: SF


stream_mode: updates
content: {'tools': {'messages': [ToolMessage(content="It's always sunny